# A Small Chemistry Database

This notebook is a side-kick to `11-pandas.ipynb`. There we used pandas on spectra;
here we use it on something every chemist already has on a shelf somewhere: a list of
compounds with their names, CAS numbers and structures.

The point is that a `DataFrame` is not restricted to numbers. A column can hold
molecules, and with [RDKit](https://www.rdkit.org/docs/index.html) those molecules are
*drawn* right inside the table. Once your compounds live in a `DataFrame` you get
lookup, filtering and plotting for free.

## Learning objectives

By the end of this notebook you will be able to:

- read a compound table (name, CAS, SMILES) into a `DataFrame`
- turn a SMILES column into real molecules and see the structures in the table
- look up a compound with `.loc` by name, and pick single values out of a row
- compute molecular properties as new columns with `.apply()`
- search the table by substructure using SMARTS
- plot one computed property against another

### Environment setup

`rdkit` and `pooch` come with the course environment (`uv sync`), so locally there is
nothing to do. On Colab we have to install them first.

In [ ]:
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")

In [ ]:
if IN_COLAB:
    %pip install -q rdkit pooch pandas matplotlib

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from pooch import retrieve
from rdkit import Chem
from rdkit.Chem import Descriptors, PandasTools
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

repo = "https://raw.githubusercontent.com/luchem/KEMM30/master/"

## The data

Our database is an ordinary CSV file with three columns:

* `name` &ndash; the trivial name we actually use in the lab
* `cas` &ndash; the [CAS registry number](https://en.wikipedia.org/wiki/CAS_Registry_Number),
  a unique identifier for a substance
* `smiles` &ndash; a [SMILES](https://en.wikipedia.org/wiki/SMILES) string, which is the
  structure written as text. `CCO` is ethanol, `c1ccccc1` is benzene.

Note the CAS numbers are read as *strings*, not numbers &ndash; `50-78-2` is a label, and
the leading zero in a number like `00050-78-2` would be lost if pandas treated it as
an integer.

In [ ]:
# Locally the file sits next to the notebook; on Colab we download it.
csv_file = "Data/compounds.csv"
if not os.path.exists(csv_file):
    csv_file = retrieve(
        f"{repo}lectures/Data/compounds.csv",
        known_hash="414feb7103cf92f5a401eaeda487dbea75805a2e084364db26d6cf101879c12a",
    )

df = pd.read_csv(csv_file)
df

In [ ]:
print(df.shape)  # (rows, columns)
df.dtypes

## Drawing the structures

`PandasTools` reads a SMILES column and adds a column of RDKit molecules. Because it
also teaches pandas how to display molecules, the table now shows the drawn structure
instead of a memory address.

In [ ]:
PandasTools.AddMoleculeColumnToFrame(df, smilesCol="smiles", molCol="structure")
PandasTools.RenderImagesInAllDataFrames(images=True)  # also draw in slices of df
PandasTools.molSize = (200, 150)

df

What sits in the `structure` column is not a picture and not a string &ndash; it is a
`Chem.Mol` object, a real molecule you can ask questions:

In [ ]:
mol = df.loc[10, "structure"]
print(type(mol))
print("atoms (heavy):", mol.GetNumAtoms())
print("rings:", mol.GetRingInfo().NumRings())
mol

## Looking things up with `.loc`

Right now the rows are numbered `0, 1, 2, ...`, which is not how a chemist thinks. Let
us index the table by compound name instead, so we can ask for a compound the way we
would ask a colleague.

In [ ]:
df = df.set_index("name")
df.head()

`.loc[label]` returns one row as a `Series`:

In [ ]:
df.loc["aspirin"]

A `Series` is a one-dimensional object, so it prints as text and the structure is not
drawn. Ask for a *list* of labels instead and you get a `DataFrame` back &ndash; with the
picture:

In [ ]:
df.loc[["aspirin"]]

Add a second argument to pick a single value out of the row. This is the everyday
lookup: *what is the CAS number of paracetamol?*

In [ ]:
df.loc["paracetamol", "cas"]

In [ ]:
# several compounds, selected columns
df.loc[["aspirin", "caffeine", "vanillin"], ["cas", "structure"]]

`.loc` also slices by label, and unlike ordinary Python slicing **both ends are
included**. Sorting the index first makes the result alphabetical:

In [ ]:
df.sort_index().loc["acetone":"benzoic acid"]

Two things worth remembering:

* `.loc` works on **labels**, `.iloc` on **positions**. `df.iloc[0]` is the first row
  whatever it is called.
* Misspell a label and you get a `KeyError`. That is a feature &ndash; a silent wrong answer
  would be much worse.

In [ ]:
df.iloc[0]

### Task 1

Use `.loc` to answer these:

1. What is the CAS number of vanillin?
2. Show the rows for the three compounds you would call painkillers, with their
   structures.
3. What happens if you ask for `df.loc["asprin"]`? Read the error message.

<details>
<summary>💡 Show solution</summary>

```python
df.loc["vanillin", "cas"]

df.loc[["aspirin", "paracetamol", "ibuprofen"], ["cas", "structure"]]

df.loc["asprin"]  # KeyError: 'asprin' - the label does not exist
```

</details>

## Computing new columns

A `DataFrame` column is a `Series`, and `.apply()` runs a function on every element of
it. Since our `structure` column holds molecules, we can hand each one to an RDKit
descriptor function and collect the answers as a new column.

This is where the database starts to earn its keep: nobody typed these numbers in.

In [ ]:
df["formula"] = df["structure"].apply(CalcMolFormula)
df["mw"] = df["structure"].apply(Descriptors.MolWt)
df["logp"] = df["structure"].apply(Descriptors.MolLogP)
df["tpsa"] = df["structure"].apply(Descriptors.TPSA)
df["hbd"] = df["structure"].apply(Descriptors.NumHDonors)

df[["formula", "mw", "logp", "tpsa", "hbd"]].round(2)

`mw` is the molecular weight, `logp` an estimate of how a compound partitions between
octanol and water (high = greasy), `tpsa` the topological polar surface area and `hbd`
the number of hydrogen-bond donors.

With numbers in the table, the usual pandas tools apply. Boolean masks combine with
`.loc` to give a filtered view:

In [ ]:
df.loc[df["mw"] > 180, ["formula", "mw", "structure"]]

### Task 2

1. Add a column `heavy_atoms` counting the non-hydrogen atoms of each molecule
   (`Descriptors.HeavyAtomCount`).
2. Which compound in the table is the most water-friendly, i.e. has the lowest `logp`?
   `.idxmin()` gives you the label of the smallest value.

<details>
<summary>💡 Show solution</summary>

```python
df["heavy_atoms"] = df["structure"].apply(Descriptors.HeavyAtomCount)

df["logp"].idxmin()
```

</details>

## Searching by substructure

Names and formulas are a poor way to ask *"which of these are carboxylic acids?"*. The
chemical way is to match a fragment, written as
[SMARTS](https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html) &ndash; a pattern
language for structures, the way regular expressions are a pattern language for text.

`C(=O)[OX2H1]` reads: a carbon, double bonded to an oxygen, single bonded to an
oxygen carrying one hydrogen.

In [ ]:
acid = Chem.MolFromSmarts("C(=O)[OX2H1]")
df["is_acid"] = df["structure"].apply(lambda mol: mol.HasSubstructMatch(acid))

df.loc[df["is_acid"], ["formula", "structure"]]

The result is again just a boolean mask, so it combines with everything else. Esters
(`C(=O)O` with a carbon on the second oxygen) and aromatic rings work the same way:

In [ ]:
ester = Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")
aromatic = Chem.MolFromSmarts("c1ccccc1")

df["is_ester"] = df["structure"].apply(lambda mol: mol.HasSubstructMatch(ester))
df["is_aromatic"] = df["structure"].apply(lambda mol: mol.HasSubstructMatch(aromatic))

# aromatic esters only
df.loc[df["is_ester"] & df["is_aromatic"], ["formula", "structure"]]

### Task 3

1. Aspirin is both an ester and an acid. Verify that with a single `.loc` expression.
2. Write a SMARTS for an alcohol (`[OX2H]` bonded to an sp3 carbon: `[CX4][OX2H]`) and
   list the alcohols in the table.

<details>
<summary>💡 Show solution</summary>

```python
df.loc[df["is_ester"] & df["is_acid"], ["formula", "structure"]]

alcohol = Chem.MolFromSmarts("[CX4][OX2H]")
df["is_alcohol"] = df["structure"].apply(lambda mol: mol.HasSubstructMatch(alcohol))
df.loc[df["is_alcohol"], ["formula", "structure"]]
```

</details>

## Plotting the database

Two computed columns are enough for a chemical space plot. Molecular weight against
logP is the kind of figure you see on the first slide of every medicinal chemistry
talk.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
points = ax.scatter(df["mw"], df["logp"], c=df["tpsa"], cmap="viridis", s=60)

for name, row in df.iterrows():
    ax.annotate(
        name,
        (row["mw"], row["logp"]),
        fontsize=8,
        xytext=(5, 4),
        textcoords="offset points",
    )

ax.margins(0.15)  # room for the labels
ax.set_xlabel("molecular weight (g/mol)")
ax.set_ylabel("logP")
ax.set_title("chemical space of our little database")
fig.colorbar(points, label="TPSA (Å$^2$)")
fig.tight_layout()

### Task 4

Colour the points by whether the compound is an acid instead of by TPSA, and give the
plot a legend. Hint: plot the two subsets separately with two `ax.scatter()` calls.

<details>
<summary>💡 Show solution</summary>

```python
fig, ax = plt.subplots(figsize=(7, 5))
for is_acid, group in df.groupby("is_acid"):
    label = "acid" if is_acid else "not an acid"
    ax.scatter(group["mw"], group["logp"], s=60, label=label)
ax.set_xlabel("molecular weight (g/mol)")
ax.set_ylabel("logP")
ax.legend()
```

</details>

## Saving the database

A CSV file is text, and a molecule object is not. Writing the table straight to disk
would put unreadable rubbish in the `structure` column, so drop it &ndash; the SMILES column
already holds the same information in a form a file can carry.

In [ ]:
out = df.drop(columns="structure")
out.to_csv("compounds_with_properties.csv")

# peek at the first two lines of what we just wrote
with open("compounds_with_properties.csv") as f:
    print(f.readline())
    print(f.readline())

### Task 5

Add your own favourite compound to the database: look up its SMILES (Wikipedia lists
one for most substances), append a row with `df.loc["my compound"] = ...` or by
building a small `DataFrame` and using `pd.concat`, then re-run the structure and
descriptor cells so it is drawn and characterised like the rest.

<details>
<summary>💡 Show solution</summary>

```python
new = pd.DataFrame(
    [{"cas": "50-06-6", "smiles": "CCC1(c2ccccc2)C(=O)NC(=O)NC1=O"}],
    index=["phenobarbital"],
)
new.index.name = "name"
df = pd.concat([df, new])

PandasTools.AddMoleculeColumnToFrame(df, smilesCol="smiles", molCol="structure")
df["mw"] = df["structure"].apply(Descriptors.MolWt)
df.loc[["phenobarbital"], ["cas", "mw", "structure"]]
```

</details>

## Summary

* A `DataFrame` column can hold any Python object, molecules included. `PandasTools`
  makes those molecules draw themselves in the table.
* `.loc` is the lookup tool: by label, by list of labels, by label slice, or by
  boolean mask &ndash; optionally with a second argument to pick columns.
* `.apply()` turns any function of one molecule into a whole column of results, which
  is how the descriptors and the substructure flags were made.
* Once the properties are columns, filtering and plotting are the same pandas you
  already know from lecture 11.

### Read more

* [RDKit documentation](https://www.rdkit.org/docs/index.html) and the
  [Getting Started guide](https://www.rdkit.org/docs/GettingStartedInPython.html)
* [Daylight SMARTS theory](https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html)
* pandas cheat sheets in [`cheat_sheets/`](cheat_sheets/)